# Curanex: hidden (undiagnosed) diabetes from a routine blood test (CPU)

Replaces the Sylhet symptom check, which has no outside data. **Rules fixed before running (2026-11-12):**
- Truth: HbA1c >= 6.5% in adults (18+) never told they have diabetes. Sugar tests (HbA1c, glucose) are NOT inputs.
- Inputs: age, sex, BMI, waist, triglycerides, HDL cholesterol, ALT, GGT, uric acid, systolic blood pressure (average of readings),
  told high blood pressure, smoking now - all on a routine check-up report.
- Train NHANES 2017-18; 20% stratified validation; logistic unless boosting wins by 0.02 validation AUC; threshold = highest with
  validation sensitivity >= 0.85. **Outside test, scored once:** NHANES 2021-23 (different people, post-pandemic cycle).
- Verified if test sensitivity and AUC are both >= 0.80; live at >= 0.75; otherwise off.

In [ ]:
import io, json, urllib.request
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
YEAR = {"J": 2017, "L": 2021}


def nh(name, c):
    url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{YEAR[c]}/DataFiles/{name}_{c}.xpt"
    try:
        return pd.read_sas(io.BytesIO(urllib.request.urlopen(url, timeout=180).read()), format="xport")
    except Exception as e:
        print("missing", name, c, e); return pd.DataFrame({"SEQN": []})


def cycle(c):
    d = nh("DEMO", c)[["SEQN", "RIAGENDR", "RIDAGEYR"]]
    for f in ("BMX", "BIOPRO", "HDL", "GHB", "DIQ", "BPQ", "BPXO", "SMQ"):
        d = d.merge(nh(f, c), on="SEQN", how="left")
    col = lambda k: d[k] if k in d else pd.Series(np.nan, index=d.index)
    told = col("DIQ010").map({1.0: 1.0, 2.0: 0.0, 3.0: 0.0})
    f = pd.DataFrame({"age": d.RIDAGEYR, "male": d.RIAGENDR.map({1.0: 1.0, 2.0: 0.0}), "bmi": col("BMXBMI"), "waist_cm": col("BMXWAIST"),
                      "triglycerides": col("LBXSTR"), "hdl": col("LBDHDD"), "alt": col("LBXSATSI"), "ggt": col("LBXSGTSI"), "uric_acid": col("LBXSUA"),
                      "systolic_bp": d[[k for k in ("BPXOSY1", "BPXOSY2", "BPXOSY3") if k in d]].mean(axis=1),
                      "high_bp": col("BPQ020").map({1.0: 1.0, 2.0: 0.0}),
                      "smoker_current": np.where(col("SMQ020") == 2, 0.0, col("SMQ040").map({1.0: 1.0, 2.0: 1.0, 3.0: 0.0}))})
    f["y"] = np.where(col("LBXGH").notna() & (told == 0) & (d.RIDAGEYR >= 18), (col("LBXGH") >= 6.5).astype(float), np.nan)
    return f[f.y.notna()]


def report(y, p, t):
    fl = p >= t
    r = {"n": int(len(y)), "positives": int(y.sum()), "sensitivity": round(float(fl[y == 1].mean()), 3),
         "specificity": round(float((~fl[y == 0]).mean()), 3), "auc": round(float(roc_auc_score(y, p)), 3)}
    low = min(r["sensitivity"], r["auc"]); r["status"] = "verified" if low >= 0.80 else "live" if low >= 0.75 else "off"
    return r


a, b = cycle("J"), cycle("L")
feats = [k for k in a.columns if k != "y"]
print("train 2017-18", len(a), int(a.y.sum()), "| test 2021-23", len(b), int(b.y.sum()), {k: round(float(b[k].isna().mean()), 2) for k in feats})
x, y = a[feats].to_numpy(np.float32), a.y.to_numpy(int)
tr, va = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
fits = {"logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=3000)).fit(x[tr], y[tr]),
        "boosting": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=300, random_state=42).fit(x[tr], y[tr])}
aucs = {k: roc_auc_score(y[va], m.predict_proba(x[va])[:, 1]) for k, m in fits.items()}
name = "boosting" if aucs["boosting"] > aucs["logistic"] + 0.02 else "logistic"; model = fits[name]
pv = model.predict_proba(x[va])[:, 1]
thr = float(max(v for v in np.unique(np.round(pv, 5)) if (pv[y[va] == 1] >= v).mean() >= 0.85))
res = report(b.y.to_numpy(int), model.predict_proba(b[feats].to_numpy(np.float32))[:, 1], thr)
print(f"OUTSIDE TEST on NHANES 2021-23 (once): hidden diabetes {name} val AUC {aucs[name]:.3f} -> {res}")
meta = {"target": "hidden_diabetes_labs", "features": feats, "model": name, "threshold": thr, "min_age": 18,
        "metrics": {"validation_auc": round(aucs[name], 3), "test": res}, "trained_on": f"US CDC NHANES 2017-18, {len(tr):,} people",
        "test_set": "NHANES 2021-23, different people", "median": np.nanmedian(x[tr], axis=0).tolist(),
        "train_min": np.nanmin(x[tr], axis=0).tolist(), "train_max": np.nanmax(x[tr], axis=0).tolist()}
if name == "logistic":
    imp, sc, clf = (model.steps[i][1] for i in range(3))
    meta |= {"impute": imp.statistics_.tolist(), "mean": sc.mean_.tolist(), "scale": sc.scale_.tolist(), "coef": clf.coef_[0].tolist(), "intercept": float(clf.intercept_[0])}
else:
    meta |= {"trees": [{k: p[0].nodes[k].tolist() for k in ("feature_idx", "num_threshold", "left", "right", "is_leaf", "value", "missing_go_to_left")} for p in model._predictors],
             "baseline": float(np.ravel(model._baseline_prediction)[0])}
json.dump(meta, open("/kaggle/working/hidden_diabetes_labs.json", "w"))